# DoorDash SQL Interview Prep

This notebook uses **DuckDB** as an in-memory SQL engine and **jupysql** for SQL magic cells.

- `%%sql` — run a full SQL block
- `%sql` — run a single-line SQL query
- Regular Python cells work as normal

---

## Two Modes (active across all prep sessions)

### Mode 1 — Interviewer
Claude asks the question. You can ask clarifying questions and Claude responds as the interviewer would. Work your solution in the cells below the question.

**Activate:** Tell Claude "Mode 1" or "ask me a question"

### Mode 2 — Answer Checker
Once your solution is complete, tell Claude "Mode 2" or "check my answer". Claude will read your solution, rate it out of 10, explain the score, and give concrete improvement suggestions.

**Activate:** Tell Claude "Mode 2" or "check my answer" / "solution done"

---

## Setup — Run this first

In [2]:
%load_ext sql
%sql duckdb:///:memory:
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

import pandas as pd
import numpy as np
print('Ready!')

Connecting to 'duckdb:///:memory:'

Ready!


---
## Q1 — Top Dashers by Avg Delivery Time per City

For each city, return the **top 2 dashers** by average delivery time (fastest first).
Include: `city`, `dasher_name`, `avg_delivery_minutes` (rounded to 1 decimal), and their `rank` within the city.
If two dashers have the same average delivery time, they should share a rank.

### Schema
```
dashers    (dasher_id, dasher_name, city, signup_date)
deliveries (delivery_id, dasher_id, restaurant_id, picked_up_at TIMESTAMP, delivered_at TIMESTAMP, delivery_date)
```

In [3]:
%%sql
-- Sample data setup
CREATE OR REPLACE TABLE dashers (
    dasher_id   INTEGER,
    dasher_name VARCHAR,
    city        VARCHAR,
    signup_date DATE
);

INSERT INTO dashers VALUES
    (1, 'Ana',  'SF',  '2023-01-01'),
    (2, 'Ben',  'SF',  '2023-02-01'),
    (3, 'Cara', 'SF',  '2023-03-01'),
    (4, 'Dev',  'NYC', '2023-01-15'),
    (5, 'Elle', 'NYC', '2023-02-15');

CREATE OR REPLACE TABLE deliveries (
    delivery_id   INTEGER,
    dasher_id     INTEGER,
    restaurant_id INTEGER,
    picked_up_at  TIMESTAMP,
    delivered_at  TIMESTAMP,
    delivery_date DATE
);

INSERT INTO deliveries VALUES
    (1, 1, 101, '2024-01-01 10:00', '2024-01-01 10:28', '2024-01-01'),
    (2, 1, 102, '2024-01-01 12:00', '2024-01-01 12:22', '2024-01-01'),
    (3, 2, 101, '2024-01-01 11:00', '2024-01-01 11:35', '2024-01-01'),
    (4, 2, 103, '2024-01-01 13:00', '2024-01-01 13:40', '2024-01-01'),
    (5, 3, 102, '2024-01-01 09:00', '2024-01-01 09:18', '2024-01-01'),
    (6, 4, 104, '2024-01-01 10:00', '2024-01-01 10:30', '2024-01-01'),
    (7, 5, 104, '2024-01-01 11:00', '2024-01-01 11:25', '2024-01-01'),
    (8, 5, 105, '2024-01-01 14:00', '2024-01-01 14:20', '2024-01-01');

,Success


In [4]:
%%sql
-- Your solution here
select 
    dashers.city
    , dashers.dasher_id
    , dashers.dasher_name
    , avg(date_diff('minute', deliveries.picked_up_at, deliveries.delivered_at)) as avg_delivery_time_minutes
    , dense_rank() over (partition by dashers.city order by avg_delivery_time_minutes) as city_dasher_rank
from dashers
inner join deliveries
on dashers.dasher_id = deliveries.dasher_id
group by 1,2,3
qualify city_dasher_rank < 3



,city,dasher_id,dasher_name,avg_delivery_time_minutes,city_dasher_rank
0,NYC,5,Elle,22.5,1
1,NYC,4,Dev,30.0,2
2,SF,3,Cara,18.0,1
3,SF,1,Ana,25.0,2


---
## Q2 — 30-Day Retention: Did Consumers Come Back?

For each consumer, find their first order date. Then determine whether they placed a second order within 30 days of that first order.

Return a summary with two columns:
- `came_back` — `'yes'` or `'no'`
- `consumer_count` — number of consumers in each group

Order by `came_back` descending.

### Schema
```
orders (order_id, consumer_id, restaurant_id, order_date DATE, order_total DECIMAL)
```

In [5]:
%%sql
CREATE OR REPLACE TABLE orders (
    order_id      INTEGER,
    consumer_id   INTEGER,
    restaurant_id INTEGER,
    order_date    DATE,
    order_total   DECIMAL
);

INSERT INTO orders VALUES
    (1,  101, 201, '2024-01-05', 25.00),
    (2,  102, 202, '2024-01-08', 30.00),
    (3,  103, 201, '2024-01-10', 18.00),
    (4,  101, 203, '2024-01-20', 22.00),
    (5,  102, 202, '2024-02-20', 28.00),
    (6,  104, 204, '2024-02-02', 15.00),
    (7,  101, 201, '2024-01-30', 35.00),
    (8,  103, 203, '2024-02-05', 20.00),
    (9,  104, 204, '2024-02-10', 40.00),
    (10, 105, 205, '2024-02-15', 12.00);

,Success


In [6]:
%%sql
-- Your solution here
with first_order as (
    select 
        consumer_id
        , min(order_date) as first_order_date
    from orders
    group by consumer_id
), cte as (
    select 
        first_order.consumer_id
        , case when orders.consumer_id is not null then 'yes' else 'no' end as made_second_order_within_30_days
    from first_order
    left join orders
    on first_order.consumer_id = orders.consumer_id
    and orders.order_date > first_order.first_order_date
    and orders.order_date <= first_order.first_order_date + INTERVAL 30 DAY
)

select 
    made_second_order_within_30_days
    , count(distinct consumer_id) as num_consumers
from cte
group by 1


,made_second_order_within_30_days,num_consumers
0,no,2
1,yes,3


---
## Q3 — A/B Test: Checkout UI Conversion Rate

DoorDash ran a 7-day A/B test on a new checkout page design. Users were randomly assigned to `control` (old UI) or `treatment` (new UI) at the start of a session. An "order" during that window counts as a conversion for the user's assigned variant.

Write a query that returns, for each variant:
- `variant`
- `total_users` — distinct users assigned to that variant
- `converted_users` — distinct users who placed at least one order during the experiment window
- `conversion_rate` — rounded to 4 decimal places
- `lift_vs_control` — for treatment, the absolute difference in conversion rate vs. control; for control, show `NULL`

The experiment ran from `2024-03-01` to `2024-03-07` inclusive. Only count orders placed within that window.

### Schema
```
experiment_assignments (user_id, variant, assigned_date DATE)
orders                 (order_id, user_id, order_date DATE, order_total DECIMAL)
```

In [9]:
%%sql
CREATE OR REPLACE TABLE experiment_assignments (
    user_id       INTEGER,
    variant       VARCHAR,
    assigned_date DATE
);

INSERT INTO experiment_assignments VALUES
    (1001, 'control',   '2024-03-01'),
    (1002, 'control',   '2024-03-01'),
    (1003, 'control',   '2024-03-02'),
    (1004, 'control',   '2024-03-02'),
    (1005, 'control',   '2024-03-03'),
    (1006, 'treatment', '2024-03-01'),
    (1007, 'treatment', '2024-03-01'),
    (1008, 'treatment', '2024-03-02'),
    (1009, 'treatment', '2024-03-03'),
    (1010, 'treatment', '2024-03-03');

CREATE OR REPLACE TABLE ab_orders (
    order_id    INTEGER,
    user_id     INTEGER,
    order_date  DATE,
    order_total DECIMAL
);

INSERT INTO ab_orders VALUES
    (301, 1001, '2024-03-03', 22.00),
    (302, 1003, '2024-03-04', 18.50),
    (303, 1005, '2024-03-06', 31.00),
    (304, 1006, '2024-03-02', 27.00),
    (305, 1007, '2024-03-03', 15.00),
    (306, 1008, '2024-03-04', 40.00),
    (307, 1009, '2024-03-05', 12.00),
    -- user 1002, 1004 never ordered (control no-converts)
    -- user 1010 ordered outside window — should NOT count
    (308, 1010, '2024-03-10', 20.00);

,Success


In [15]:
%%sql
-- Your solution here
with temp_cte as (
    select 
        ea.variant
        , count(distinct ea.user_id) as total_users
        , count(distinct case when abo.user_id is not null then abo.user_id end) as converted_users
        , round(converted_users/total_users, 4) as conversion_rate
    from experiment_assignments ea
    left join ab_orders abo
    on ea.user_id = abo.user_id
    and ea.assigned_date <= abo.order_date
    and abo.order_date between '2024-03-01' and '2024-03-08'
    where ea.assigned_date between '2024-03-01' and '2024-03-08'
    group by 1
), ctrl_val as (
    select *
    from temp_cte
    where variant = 'control'
)

select 
    temp_cte.*
    , case when temp_cte.variant = 'control' then null
        else round(temp_cte.conversion_rate - ctrl_val.conversion_rate, 4)
        end as lift_vs_control
from temp_cte, ctrl_val




,variant,total_users,converted_users,conversion_rate,lift_vs_control
0,control,5,3,0.6,NaN
1,treatment,5,4,0.8,0.2


---
## Q4 — Merchant Revenue Trends (Window Functions, Multi-Part)

You have a table of orders placed on DoorDash. Analyze merchant performance over time.

**Part A:** For each merchant, compute their **total revenue per month** and their **running cumulative revenue** (ordered by month). Return `merchant_id`, `merchant_name`, `month`, `monthly_revenue`, and `cumulative_revenue`.

**Part B:** Within each **city and month**, rank merchants by their monthly revenue (highest = rank 1). Return only the **top 2 merchants per city per month**. Include ties.

**Part C:** Identify merchants who had a **month-over-month revenue decline** — revenue in a given month was lower than the previous month. Return the merchant, the month of the decline, the previous month's revenue, and the current month's revenue.

### Schema
```
orders
------
order_id      INT
merchant_id   INT
merchant_name VARCHAR
city          VARCHAR
order_date    DATE
order_total   DECIMAL(10,2)
```

In [16]:
%%sql
DROP TABLE IF EXISTS orders;
CREATE TABLE orders (
    order_id      INT,
    merchant_id   INT,
    merchant_name VARCHAR(100),
    city          VARCHAR(50),
    order_date    DATE,
    order_total   DECIMAL(10,2)
);

INSERT INTO orders VALUES
-- Merchant 1: Burger Barn (SF)
(1,  1, 'Burger Barn',   'SF', '2024-01-05', 45.00),
(2,  1, 'Burger Barn',   'SF', '2024-01-18', 30.00),
(3,  1, 'Burger Barn',   'SF', '2024-02-10', 90.00),
(4,  1, 'Burger Barn',   'SF', '2024-02-22', 60.00),
(5,  1, 'Burger Barn',   'SF', '2024-03-08', 40.00),
-- Merchant 2: Taco Town (SF)
(6,  2, 'Taco Town',     'SF', '2024-01-03', 80.00),
(7,  2, 'Taco Town',     'SF', '2024-01-25', 70.00),
(8,  2, 'Taco Town',     'SF', '2024-02-14', 50.00),
(9,  2, 'Taco Town',     'SF', '2024-03-01', 120.00),
(10, 2, 'Taco Town',     'SF', '2024-03-19', 80.00),
-- Merchant 3: Pizza Palace (SF)
(11, 3, 'Pizza Palace',  'SF', '2024-01-11', 65.00),
(12, 3, 'Pizza Palace',  'SF', '2024-02-08', 110.00),
(13, 3, 'Pizza Palace',  'SF', '2024-03-15', 55.00),
-- Merchant 4: Noodle Nook (LA)
(14, 4, 'Noodle Nook',   'LA', '2024-01-07', 95.00),
(15, 4, 'Noodle Nook',   'LA', '2024-01-20', 85.00),
(16, 4, 'Noodle Nook',   'LA', '2024-02-05', 70.00),
(17, 4, 'Noodle Nook',   'LA', '2024-03-10', 100.00),
-- Merchant 5: Sushi Spot (LA)
(18, 5, 'Sushi Spot',    'LA', '2024-01-15', 130.00),
(19, 5, 'Sushi Spot',    'LA', '2024-02-20', 90.00),
(20, 5, 'Sushi Spot',    'LA', '2024-02-28', 60.00),
(21, 5, 'Sushi Spot',    'LA', '2024-03-05', 70.00);

,Success


In [19]:
%%sql
-- Part A: Monthly revenue + running cumulative revenue per merchant
-- Your solution here
select 
    merchant_id
    , merchant_name
    , date_trunc('month', order_date) as month
    , sum(order_total) as monthly_revenue
    , sum(monthly_revenue) over(partition by merchant_id order by month) as cumulative_revenue
from orders
group by 1,2,3
order by 1,2,3


,merchant_id,merchant_name,month,monthly_revenue,cumulative_revenue
0,1,Burger Barn,2024-01-01,75.0,75.0
1,1,Burger Barn,2024-02-01,150.0,225.0
2,1,Burger Barn,2024-03-01,40.0,265.0
3,2,Taco Town,2024-01-01,150.0,150.0
4,2,Taco Town,2024-02-01,50.0,200.0
5,2,Taco Town,2024-03-01,200.0,400.0
6,3,Pizza Palace,2024-01-01,65.0,65.0
7,3,Pizza Palace,2024-02-01,110.0,175.0
8,3,Pizza Palace,2024-03-01,55.0,230.0
9,4,Noodle Nook,2024-01-01,180.0,180.0


In [20]:
%%sql
-- Part B: Top 2 merchants by monthly revenue within each city (with ties)
-- Your solution here
with temp_cte as (
    select 
        city
        , date_trunc('month', order_date) as month
        , merchant_id
        , merchant_name
        , sum(order_total) as monthly_revenue
        , dense_rank() over(partition by city, month order by monthly_revenue desc) as city_monthly_rank
    from orders
    group by 1,2,3,4
)

select *
from temp_cte
where city_monthly_rank <= 2

,city,month,merchant_id,merchant_name,monthly_revenue,city_monthly_rank
0,LA,2024-03-01,4,Noodle Nook,100.0,1
1,LA,2024-03-01,5,Sushi Spot,70.0,2
2,SF,2024-03-01,2,Taco Town,200.0,1
3,SF,2024-03-01,3,Pizza Palace,55.0,2
4,LA,2024-01-01,4,Noodle Nook,180.0,1
5,LA,2024-01-01,5,Sushi Spot,130.0,2
6,LA,2024-02-01,5,Sushi Spot,150.0,1
7,LA,2024-02-01,4,Noodle Nook,70.0,2
8,SF,2024-01-01,2,Taco Town,150.0,1
9,SF,2024-01-01,1,Burger Barn,75.0,2


In [22]:
%%sql
-- Part C: Merchants with month-over-month revenue decline
-- Your solution here
-- note, this will give all months of decline we can use rank to get first month of decline as well
with temp_cte as (
    select 
        merchant_id
        , merchant_name
        , date_trunc('month', order_date) as month
        , sum(order_total) as monthly_revenue
        , lag(monthly_revenue) over(partition by merchant_id order by month) as prev_month_revenue
        , case when monthly_revenue < prev_month_revenue then 1 else 0 end as revenue_decline_mom
    from orders
    group by 1,2,3
)

select 
    merchant_id
    , merchant_name
    , month as month_of_decline
    , monthly_revenue
    , prev_month_revenue
from temp_cte
where revenue_decline_mom = 1

,merchant_id,merchant_name,month_of_decline,monthly_revenue,prev_month_revenue
0,2,Taco Town,2024-02-01,50.0,150.0
1,1,Burger Barn,2024-03-01,40.0,150.0
2,3,Pizza Palace,2024-03-01,55.0,110.0
3,4,Noodle Nook,2024-02-01,70.0,180.0
4,5,Sushi Spot,2024-03-01,70.0,150.0
